In [1]:
import tensorflow as tf
import numpy as np
import gym
from ou_noise import OUNoise
import random


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })



def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.0001
    CRITIC_LEARNING_RATE =  0.001
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v0'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
    epochs = 15000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = DDPG(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        llegadas =0
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0 
            max_steps = 45 #es solo a efecto de contador
            r_episodio_actual = []
            #llegadas =0
            while (not done):
                #print('step =', step)
                #wait = input("PRESS ENTER TO CONTINUE.")
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)
                    #print('Explorando')

                action = np.clip(action,min_action,max_action)
                #print('ruido =', max(epsilon,0)*ruido.noise(),'epsilon =',epsilon)
                #print('LA ACCION clipeada ES', action, action.shape)
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                #print('Steps = ', info['Steps'],'V0 =',info['v0'])
                #print('normalized_state =',normalized_state)
                #wait = input("PRESS ENTER TO CONTINUE.")
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
#                 print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
#                 print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                
                step += 1
            
            
            #if info['Steps'] > max_steps:
             #   llegadas +=1
#             print ('--------------------------------------------')
#             print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
#             print ('--------------------------------------------')

                

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_DDPG01.npy',Reward_episodios)
            

            print('epoch          = ', i)      
            print('Episode Reward = ', episode_r/2)
            print('Last Reward    = ', reward)
            print('State          = ', state)
            print()

            if i % 1000 == 0:      # Save model every 1000 iterations
                ddpg.save()
            
        ddpg.save()

C:\Users\archi\anaconda3\envs\BTP\lib\site-packages\gym\logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


epoch          =  0
Episode Reward =  45.07339500000001
Last Reward    =  0.07476
State          =  [  20.71304 7775.74    1976.3385 ]

Model saved in file: actor_model
epoch          =  1
Episode Reward =  3.1998550000000012
Last Reward    =  0.07784
State          =  [  21.77833 8152.175   1566.9768 ]

epoch          =  2
Episode Reward =  60.79978499999998
Last Reward    =  0.87763
State          =  [   76.51777 27757.74      230.08496]

epoch          =  3
Episode Reward =  21.140675000000005
Last Reward    =  0.15979
State          =  [   17.663261  6628.532    -2548.009   ]

epoch          =  4
Episode Reward =  19.056185000000003
Last Reward    =  0.24405
State          =  [   51.499187 19315.074    -2889.4958  ]

epoch          =  5
Episode Reward =  34.68377
Last Reward    =  0.05867
State          =  [ 136.4464 3498.3843  519.7642]

epoch          =  6
Episode Reward =  18.750059999999994
Last Reward    =  0.04916
State          =  [   13.976994  5247.395    -1690.2336  ]

ep

epoch          =  58
Episode Reward =  7.158020011442844
Last Reward    =  3e-05
State          =  [  0.36013183 135.21953    135.21953   ]

epoch          =  59
Episode Reward =  8.263315000000004
Last Reward    =  0.00497
State          =  [    7.4939094  2812.8538    -2525.335    ]

epoch          =  60
Episode Reward =  15.466650000000005
Last Reward    =  0.62497
State          =  [   49.41972 14742.822    2463.8516 ]

epoch          =  61
Episode Reward =  51.77801
Last Reward    =  0.99557
State          =  [  130.37492 19471.762     704.9917 ]

epoch          =  62
Episode Reward =  66.83509995680095
Last Reward    =  0.7624800205230713
State          =  [   97.80562 20494.268   -2635.4766 ]

epoch          =  63
Episode Reward =  35.510725001041514
Last Reward    =  0.07051
State          =  [  18.859356 7079.972     416.34677 ]

epoch          =  64
Episode Reward =  39.090135010710945
Last Reward    =  0.5693399906158447
State          =  [  53.766808 7075.4233   2357.2656  

c:\users\archi\python code\btp\luis_upgrade\mppt-gym_v4\gym_mppt\envs\mppt_env_shaded0.py:37: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)


epoch          =  94
Episode Reward =  44.33113001662854
Last Reward    =  0.041999999433755875
State          =  [  200.79572  5770.7417  -5665.563  ]

epoch          =  95
Episode Reward =  21.974345026988985
Last Reward    =  0.7225800156593323
State          =  [   65.74372 15553.301    1039.9082 ]

epoch          =  96
Episode Reward =  53.752684923062326
Last Reward    =  0.9950500130653381
State          =  [  106.023674 27931.838      319.63867 ]

epoch          =  97
Episode Reward =  23.1783550017482
Last Reward    =  0.04580000042915344
State          =  [ 103.10792 3643.0237  -161.04248]

epoch          =  98
Episode Reward =  30.844300000000022
Last Reward    =  0.72584
State          =  [   82.194244 30399.201     1442.762   ]

epoch          =  99
Episode Reward =  21.525050005861615
Last Reward    =  0.44635
State          =  [   49.073498 11609.739     1686.3516  ]

epoch          =  100
Episode Reward =  3.651384999860864
Last Reward    =  0.03006
State          =  [ 

...

epoch          =  14772
Episode Reward =  2.8998850025272986
Last Reward    =  0.04092999920248985
State          =  [  24.338472 6012.2207   -830.3052  ]

epoch          =  14773
Episode Reward =  88.6928251623869
Last Reward    =  0.9537500143051147
State          =  [   93.958275 29159.18       273.25977 ]

epoch          =  14774
Episode Reward =  6.442234983402938
Last Reward    =  0.04391000047326088
State          =  [   10.229104  3831.945    -1116.5818  ]

epoch          =  14775
Episode Reward =  22.431255020741226
Last Reward    =  0.20381000638008118
State          =  [   24.7164  9184.552  -2188.5098]

epoch          =  14776
Episode Reward =  40.76785488146945
Last Reward    =  0.5809800028800964
State          =  [   33.969166 11447.87      2522.2988  ]

epoch          =  14777
Episode Reward =  51.564260008744895
Last Reward    =  0.5561500191688538
State          =  [   47.745804 17503.05     -1538.916   ]

epoch          =  14778
Episode Reward =  89.34114502556622
La

epoch          =  14825
Episode Reward =  86.57139991526492
Last Reward    =  0.943560004234314
State          =  [  143.47905 43767.01     -323.25   ]

epoch          =  14826
Episode Reward =  20.406840015434625
Last Reward    =  0.2626500129699707
State          =  [   38.44471  11485.853      -30.374023]

epoch          =  14827
Episode Reward =  19.59724493563408
Last Reward    =  0.18038000166416168
State          =  [  12.87504 4793.067   -822.06934]

epoch          =  14828
Episode Reward =  4.580215003580176
Last Reward    =  0.009750000201165676
State          =  [    6.03454  2265.2126  -2152.0916 ]

epoch          =  14829
Episode Reward =  87.1519950828515
Last Reward    =  0.8908399939537048
State          =  [  105.01453 30400.305     300.54297]

epoch          =  14830
Episode Reward =  92.96973998844624
Last Reward    =  0.953790009021759
State          =  [ 6.8483315e+01  2.4347920e+04 -2.2484375e+01]

epoch          =  14831
Episode Reward =  36.880100181164295
Last 

epoch          =  14878
Episode Reward =  65.78535990230739
Last Reward    =  0.8217700123786926
State          =  [   46.810963 16980.342      701.4346  ]

epoch          =  14879
Episode Reward =  93.39038993045688
Last Reward    =  0.9522299766540527
State          =  [   68.39409 24335.574    -372.26172]

epoch          =  14880
Episode Reward =  10.194095043465495
Last Reward    =  0.11905000358819962
State          =  [  15.005123 5498.8994   -676.1177  ]

epoch          =  14881
Episode Reward =  61.16720006708056
Last Reward    =  0.6561200022697449
State          =  [   71.82566 23941.264     373.50586]

epoch          =  14882
Episode Reward =  3.498159989823762
Last Reward    =  0.025369999930262566
State          =  [   11.060504  4139.229    -1781.2188  ]

epoch          =  14883
Episode Reward =  58.95001002121717
Last Reward    =  0.7034500241279602
State          =  [   46.489452 17063.477     1599.4199  ]

epoch          =  14884
Episode Reward =  23.1673899692557
Last

epoch          =  14931
Episode Reward =  47.07658982463181
Last Reward    =  0.42642998695373535
State          =  [  26.393501 9807.691     755.8047  ]

epoch          =  14932
Episode Reward =  86.72526508849114
Last Reward    =  0.8939099907875061
State          =  [   92.57118 27037.86    -1548.9121 ]

epoch          =  14933
Episode Reward =  11.432075017881289
Last Reward    =  0.07478000223636627
State          =  [    9.97632  3738.5146  -2081.538  ]

epoch          =  14934
Episode Reward =  86.90269488035698
Last Reward    =  0.9740899801254272
State          =  [   80.39059 28166.662    -178.23633]

epoch          =  14935
Episode Reward =  24.170875079464167
Last Reward    =  0.22756999731063843
State          =  [   46.525166 17150.656     -820.0762  ]

epoch          =  14936
Episode Reward =  93.27204976664544
Last Reward    =  0.6616799831390381
State          =  [   81.02167  19438.27       -61.128906]

epoch          =  14937
Episode Reward =  24.27059494284913
Last 

epoch          =  14984
Episode Reward =  58.39960003271699
Last Reward    =  0.8104299902915955
State          =  [   34.304314 11918.845     2850.5742  ]

epoch          =  14985
Episode Reward =  91.73995990928961
Last Reward    =  0.9747899770736694
State          =  [  119.38719 41816.254    -303.54688]

epoch          =  14986
Episode Reward =  80.25073016854003
Last Reward    =  0.8021100163459778
State          =  [   94.15116 29173.252    -717.5508 ]

epoch          =  14987
Episode Reward =  11.800780020232498
Last Reward    =  0.13056999444961548
State          =  [   39.809727 10737.99     -1061.5361  ]

epoch          =  14988
Episode Reward =  9.372910020763811
Last Reward    =  0.1107499971985817
State          =  [  20.615517 6206.1094    -13.618164]

epoch          =  14989
Episode Reward =  91.038294875063
Last Reward    =  0.9999899864196777
State          =  [   71.204704 23976.03       683.9414  ]

epoch          =  14990
Episode Reward =  46.82922496274114
Last Re

In [2]:
# 7500  -> 100 minutes (1 hour 40 mins)
# 15000 -> 200 minutes (3 hours 20 mins)